In [ ]:
'''
당장은 테스트케이스 생성이 급한 게 아니라 문제 번역이 더 급함

In [ ]:
'''
새로운 코드를 작성하고 싶어. 주어진 문제로부터 입력 테스트케이스를 새롭게 생성해서 저장하는 코드야. 일단 읽어보고 모르겠는 부분은 질문해줘
1. validated_problems.csv를 한 행씩 읽는다. 이때, 위에서부터 몇 번째(id 말고, 테이블 기준 상단 첫 번째=1번째)부터 몇 번째까지의 문제에 대해서만 작업할 지 정하는 변수를 처음에 만든다. 이후 분산작업을 위한 변수임. 그리고 해당 범위의 각 문제에 대해서 아까 위에서 얘기했던 4종류의 입력 테스트 케이스를 똑같은 api_key로, gemma 4 31B it 모델에게 요청해서 총 50개 받아오게 한다(정답성 검증=20개, 나머지는 10개씩). 이때, 프롬프트는 영어로 작성하며 이전 코드에서 작성했던 4종류에 대한 설명을 참고한다. 출력 결과는 testcase_order(1~50으로 할당), type(4종류 중 하나의 이름), input(생성된 입력 문자열)이 포함된 json 형식이 되도록 강제한다. 생성이 되면 각 input들을 validated_problems의 solution열값에 해당하는 파이썬 코드를 실행해 실제로 넣어보고, 그 결과값을 output 변수에 저장한다. 만약 실행 결과 에러 혹은 시간초과(10초 이상 걸림)가 발생하는 경우라면 output은 #ERROR 라고 저장한다. llm이 출력한 해당 정보들에 problem_id까지 추가해서 generated_testcases.csv로 저장할 것이다. problem_id(원래 문제의 id), testcase_order, type, input, output으로 저장한다. 이때, generated_testcases.csv는 각 문제에 대한 작업이 끝날 때마다 저장한다. 그리고 각 문제에 대한 작업이 끝날 때마다 총 몇 개의 테스트케이스 중에 몇 개가 #ERROR가 났는지도 보고하며, 해당 문제를 처리하는데 총 몇 초가 걸렸는지도 보고한다. llm에게 요청했는데 요청 에러가 나면 약간 기다리는 방어로직을 아까 답변에서 너가 작성한 것처럼 작성하면 됨. 최대 3번 재시도하도록.

In [ ]:
import os
api_key = os.environ.get("GEMINI_API_KEY")

In [2]:
import os
import sys
import time
import json
import random
import pandas as pd
import subprocess
from google import genai
from google.genai import types

# ---------------------------------------------------------
# [설정값]
# ---------------------------------------------------------
# api_key 변수는 이미 선언되어 있다고 가정합니다. (예: api_key = "AIzaSy...")
client = genai.Client(api_key=api_key)
MODEL_NAME = 'gemma-4-31b-it'

INPUT_FILE = 'validated_problems.csv'
OUTPUT_FILE = 'generated_testcases.csv'

# [분산 처리용 범위 설정] (테이블 기준 상단 첫 번째 = 1)
START_IDX = 1       # 시작 문제 순서
END_IDX = 5000       # 끝 문제 순서 (예시로 100까지)

# ---------------------------------------------------------
# [1. 파이썬 코드 실행용 래퍼 스크립트 생성]
# ---------------------------------------------------------
runner_code = """
import sys

try:
    with open(sys.argv[1], 'r', encoding='utf-8') as f:
        code = f.read()
    exec(code, {'__name__': '__main__'})
except Exception:
    sys.exit(1)
"""

with open('_runner.py', 'w', encoding='utf-8') as f:
    f.write(runner_code)

# ---------------------------------------------------------
# [2. 유틸리티 함수]
# ---------------------------------------------------------
def generate_inputs_with_llm(problem_context, tc_type, tc_desc, count, start_order, max_retries=3):
    """
    LLM을 호출하여 특정 타입의 테스트케이스를 지정된 개수만큼 JSON 형태로 받아옵니다.
    """
    prompt = f"""
[Problem Description]
{problem_context}

Your task is to generate EXACTLY {count} test case inputs of type "{tc_type}" for the algorithm problem described above.
Type "{tc_type}" means: {tc_desc}

The test cases must be formatted as raw strings exactly as they would be provided via standard input (stdin).

Return the result STRICTLY as a JSON array of exactly {count} objects in the following format. 
Do NOT include markdown tags like ```json or any other text.
[
  {{"testcase_order": {start_order}, "type": "{tc_type}", "input": "..."}},
  {{"testcase_order": {start_order + 1}, "type": "{tc_type}", "input": "..."}},
  ...
  {{"testcase_order": {start_order + count - 1}, "type": "{tc_type}", "input": "..."}}
]
"""
    
    for attempt in range(max_retries):
        try:
            # 50개 -> 분할 요청으로 출력량이 줄었으므로 JSON 강제 모드 복구
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.7
                )
            )
            
            text = response.text.strip()
            
            # 마크다운 찌꺼기 방어 로직 (코드 블록 깨짐 방지를 위해 변수 사용)
            md_marker = "`" * 3
            if text.startswith(f'{md_marker}json'):
                text = text.replace(f'{md_marker}json', '').replace(md_marker, '').strip()
            elif text.startswith(md_marker):
                text = text.replace(md_marker, '').strip()
                
            parsed_json = json.loads(text)
            
            if len(parsed_json) > 0:
                # 지정된 개수만큼만 자르고, 혹시 모를 에러 방지를 위해 python 단에서 타입과 오더를 다시 한번 확실히 덮어씌움
                parsed_json = parsed_json[:count]
                for i, item in enumerate(parsed_json):
                    item['testcase_order'] = start_order + i
                    item['type'] = tc_type
                return parsed_json
            else:
                raise ValueError("JSON array is empty.")
                
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"        ❌ [LLM 실패] '{tc_type}' 타입 생성 중 최대 재시도 횟수 초과. 에러: {e}")
                # 실패 시 빈 리스트가 아니라, 나중에 부분 재요청이 가능하도록 #ERROR 플레이스홀더를 채워서 반환
                error_placeholders = []
                for i in range(count):
                    error_placeholders.append({
                        "testcase_order": start_order + i,
                        "type": tc_type,
                        "input": "#ERROR"
                    })
                return error_placeholders
                
            sleep_time = (5 * (2 ** attempt)) + random.uniform(1, 3)
            print(f"        ⚠️ API 에러({e}). {sleep_time:.1f}초 후 재시도합니다...")
            time.sleep(sleep_time)


def execute_solution(solution_code, input_str):
    """
    솔루션 코드에 입력값을 주입하여 실행하고 결과를 반환합니다.
    """
    with open('_temp_sol.py', 'w', encoding='utf-8') as f:
        f.write(str(solution_code))
    with open('_temp_in.txt', 'w', encoding='utf-8') as f:
        f.write(str(input_str) if input_str else "")
        
    if os.path.exists('_temp_out.txt'): os.remove('_temp_out.txt')
    
    try:
        with open('_temp_in.txt', 'r', encoding='utf-8') as f_in, \
             open('_temp_out.txt', 'w', encoding='utf-8') as f_out:
            
            subprocess.run(
                [sys.executable, '_runner.py', '_temp_sol.py'],
                stdin=f_in,
                stdout=f_out,
                stderr=subprocess.DEVNULL,
                timeout=10, 
                check=True
            )
        
        with open('_temp_out.txt', 'r', encoding='utf-8', errors='ignore') as f:
            return f.read().strip()
            
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError):
        return "#ERROR"
    except Exception:
        return "#ERROR"

# ---------------------------------------------------------
# [3. 메인 파이프라인]
# ---------------------------------------------------------
print("🚀 [Step 1] 문제 및 정답 데이터 로드 중...")
df_problems = pd.read_csv(INPUT_FILE)

start_idx = max(0, START_IDX - 1)
end_idx = min(len(df_problems), END_IDX)
target_problems = df_problems.iloc[start_idx:end_idx]

print(f"✅ 총 {len(df_problems)}문제 중 {START_IDX}번째 ~ {end_idx}번째 문제(총 {len(target_problems)}개) 생성을 시작합니다.\n")

# 분할 요청할 태스크 정의 (타입명, 설명, 요청 개수)
generation_tasks = [
    ("base", "Basic examples, small inputs, manually verifiable.", 20),
    ("edge", "Boundary values, min/max, empty structures, duplicates, negative/zero, single element, etc.", 10),
    ("counter", "Cases to break incorrect implementations (overflow, index errors, wrong sorting, DP initial errors, etc.)", 10),
    ("performance", "Large inputs near the time limit, worst-case patterns, max size values.", 10)
]

for current_table_idx, row in enumerate(target_problems.itertuples(), start=START_IDX):
    prob_id = row.id
    question_text = row.question
    solution_code = row.solution
    
    print(f"\n📝 [{current_table_idx}번째 문제] Problem {prob_id} 처리 시작...")
    problem_start_time = time.time()
    
    generated_data = []
    current_order = 1
    
    # 1. LLM에게 4번 나누어 테스트케이스 생성 요청
    for tc_idx, (tc_type, tc_desc, count) in enumerate(generation_tasks, start=1):
        print(f"  - [{tc_idx}/4] API 요청 중: '{tc_type}' 타입 {count}개 생성...")
        
        partial_data = generate_inputs_with_llm(question_text, tc_type, tc_desc, count, current_order)
        generated_data.extend(partial_data)
        current_order += count
        
    print(f"  - 총 50개 생성 완료(또는 에러 처리)! 파이썬 코드를 실행하여 정답(output)을 추출합니다...")
    
    # 2. 생성된 입력값을 코드로 실행하여 결과 확보
    results_to_save = []
    error_count = 0
    
    for item in generated_data:
        tc_order = item.get('testcase_order')
        tc_type = item.get('type')
        tc_input = item.get('input', '')
        
        # LLM 에러로 인해 input이 #ERROR인 경우 실행 자체를 우회하고 output도 #ERROR 처리
        if tc_input == "#ERROR":
            tc_output = "#ERROR"
        else:
            tc_output = execute_solution(solution_code, tc_input)
        
        if tc_output == "#ERROR":
            error_count += 1
            
        results_to_save.append({
            'problem_id': prob_id,
            'testcase_order': tc_order,
            'type': tc_type,
            'input': tc_input,
            'output': tc_output
        })
    
    # 3. 해당 문제에 대한 작업이 끝나면 CSV에 누적 저장
    df_new_tcs = pd.DataFrame(results_to_save)
    file_exists = os.path.exists(OUTPUT_FILE)
    df_new_tcs.to_csv(OUTPUT_FILE, mode='a', header=not file_exists, index=False, encoding='utf-8-sig')
    
    # 4. 결과 및 소요 시간 보고
    total_generated = len(results_to_save)
    elapsed_time = time.time() - problem_start_time
    
    print(f"  ✔ [완료] 소요 시간: {elapsed_time:.1f}초")
    print(f"  📊 통계: 총 {total_generated}개 생성됨 | 정상: {total_generated - error_count}개 | #ERROR(시간초과/에러): {error_count}개")

# 임시 파일 정리
for temp_file in ['_runner.py', '_temp_sol.py', '_temp_in.txt', '_temp_out.txt']:
    if os.path.exists(temp_file):
        os.remove(temp_file)

print(f"\n🎉 설정한 범위({START_IDX}~{END_IDX}번째)의 신규 테스트케이스 생성이 완료되었습니다! 저장 파일: {OUTPUT_FILE}")

🚀 [Step 1] 문제 및 정답 데이터 로드 중...


/tmp/ipykernel_51815/3977586796.py:155: DtypeWarning: Columns (0: id, 1: count_cases, 2: count_solutions, 3: Expected Auxiliary Space, 4: starter_code, 5: picture_num, 6: Unnamed: 21, 7: Unnamed: 22, 8: Unnamed: 23, 9: Unnamed: 25, 10: Unnamed: 26, 11: Unnamed: 30, 12: Unnamed: 32, 13: Unnamed: 36, 14: Unnamed: 50, 15: Unnamed: 54, 16: Unnamed: 56, 17: Unnamed: 60, 18: Unnamed: 80, 19: Unnamed: 84, 20: Unnamed: 92, 21: Unnamed: 102, 22: Unnamed: 110, 23: Unnamed: 114, 24: Unnamed: 120, 25: Unnamed: 126, 26: Unnamed: 140, 27: Unnamed: 144, 28: Unnamed: 150, 29: Unnamed: 156, 30: Unnamed: 164, 31: Unnamed: 170, 32: Unnamed: 173) have mixed types. Specify dtype option on import or set low_memory=False.
  df_problems = pd.read_csv(INPUT_FILE)


✅ 총 4634문제 중 1번째 ~ 4634번째 문제(총 4634개) 생성을 시작합니다.


📝 [1번째 문제] Problem 2 처리 시작...
  - [1/4] API 요청 중: 'base' 타입 20개 생성...
  - [2/4] API 요청 중: 'edge' 타입 10개 생성...
        ⚠️ API 에러(500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}). 6.2초 후 재시도합니다...
  - [3/4] API 요청 중: 'counter' 타입 10개 생성...
  - [4/4] API 요청 중: 'performance' 타입 10개 생성...
  - 총 50개 생성 완료(또는 에러 처리)! 파이썬 코드를 실행하여 정답(output)을 추출합니다...
  ✔ [완료] 소요 시간: 630.2초
  📊 통계: 총 50개 생성됨 | 정상: 33개 | #ERROR(시간초과/에러): 17개

📝 [2번째 문제] Problem 11 처리 시작...
  - [1/4] API 요청 중: 'base' 타입 20개 생성...


KeyboardInterrupt: 